# **Load Model**

In [5]:
from keras.datasets import imdb
from keras.preprocessing import sequence
import tensorflow as tf
import os
import numpy as np

vocab_size = 88584
MAXLEN = 250
BATCH_SIZE = 64

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words = vocab_size)
#Preprocess data so the len(x_train[0]) = len(x_train[0]) = 250 by trimming after 250 chars or adding 0s before the data
x_train = sequence.pad_sequences(x_train, maxlen=MAXLEN)
x_test = sequence.pad_sequences(x_test, maxlen=MAXLEN)
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 32), #Embedd each number to 32 vector
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.build(input_shape=(None, MAXLEN))
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 250, 32)        │     2,834,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,843,041 (10.85 MB)

 Trainable params: 2,843,041 (10.85 MB)

 Non-trainable params: 0 (0.00 B)

# **Training Model**

> NOTE: Optimizer doesn't matter if it is atom or rmsprop



In [7]:
model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=['acc'])
history = model.fit(x_train, y_train, epochs=10, validation_split=0.2)
print(model.evaluate(x_test, y_test))

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - acc: 0.9821 - loss: 0.0575 - val_acc: 0.8798 - val_loss: 0.4125
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - acc: 0.9845 - loss: 0.0497 - val_acc: 0.8842 - val_loss: 0.4219
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - acc: 0.9888 - loss: 0.0374 - val_acc: 0.8778 - val_loss: 0.4011
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - acc: 0.9887 - loss: 0.0373 - val_acc: 0.8796 - val_loss: 0.5032
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - acc: 0.9921 - loss: 0.0294 - val_acc: 0.8732 - val_loss: 0.4506
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - acc: 0.9915 - loss: 0.0299 - val_acc: 0.8810 - val_loss: 0.5108
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - acc: 0.9937 - loss: 0.0219 - val_acc: 0.8740 - val_loss: 0.5654
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - acc: 0.9929 - loss: 0.0271 - val_acc: 0.8756 - val_loss: 0.5741
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/

# **Predictions**

In [17]:
word_index = imdb.get_word_index()

def encode_text(text):
  tokens = tf.keras.preprocessing.text.text_to_word_sequence(text)
  tokens = [word_index[word] if word in word_index else 0 for word in tokens]
  return sequence.pad_sequences([tokens], MAXLEN)[0]

text = "that movie was just amazing, so amazing"
encoded = encode_text(text)
reverse_word_index = {value: key for (key, value) in word_index.items()}
def decode_integers(integers):
  PAD=0
  text=""
  for num in integers:
    if num != PAD:
      text += reverse_word_index[num] + " "
  return text[:-1]
print(decode_integers(encoded))

def predict(text):
  encoded_text = encode_text(text)
  pred = np.zeros((1,250))
  pred[0] = encoded_text
  result = model.predict(pred)
  print(result[0])
#Removing random words adjust the prediction levels
positive_review = "That movie was so awesome! I really loved it and would watch it again because it was amazingly great"
predict(positive_review)
negative_review = "that movie sucked. I hated it and wouldn't watch it again. Was one of the worst things I've ever watched"
predict(negative_review)
#If Y -> 1, positive. If Y -> 0, negative


that movie was just amazing so amazing
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
[0.9960573]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
[0.01628531]


# **RNN Play Generator**
Character generation: Predict next word in a sentence

In [32]:
%tensorflow_version 2.x  # this line is not required unless you are in a notebook
from keras.preprocessing import sequence
import keras
import tensorflow as tf
import os
import numpy as np
from google.colab import files
#path_to_file = list(files.upload().keys())[0] #Load file from your PC
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print('Length of text: {} characters'.format(len(text)))
#print(text[:13])
#Encoding
vocab = sorted(set(text)) #Sort unique letter
char2idx = {u:i for i, u in enumerate(vocab)} #map each unique letter to frequency
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])
def int_to_text(ints):
  try:
    ints = ints.numpy()
  except:
    pass
  return ''.join(idx2char[ints])
original_text = int_to_text(text_as_int)
#print(original_text[:13]) Same print as above
seq_length = 100 #length of Training example is 100
examples_per_epoch = len(text)//(seq_length+1)
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length+1, drop_remainder=True) #Batch to 101
def split_input_target(chunk):
  input_text = chunk[:-1]
  target_text = chunk[1:]
  return input_text, target_text
dataset = sequences.map(split_input_target)
BATCH_SIZE = 64

BUFFER_SIZE = 10000
data = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.
Length of text: 1115394 characters


# **Building Model**

In [63]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
  model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.LSTM(rnn_units,
                        return_sequences=True,
                        stateful=True,
                        recurrent_initializer='glorot_uniform'),
    tf.keras.layers.Dense(vocab_size)
  ])
  return model
BATCH_SIZE = 64
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 256
RNN_UNITS = 1024
model = build_model(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS, BATCH_SIZE)
model.build(tf.TensorShape([BATCH_SIZE, None]))
model.summary()
for input_example_batch, target_example_batch in data.take(1):
  example_batch_predictions = model(input_example_batch) #Shape: (64, 100, 65)
pred = example_batch_predictions[0] #100 length
time_pred = pred[0] #65 length. Probability of every character occurring next


Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_27 (Embedding)        │ (64, None, 256)        │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_19 (LSTM)                  │ (64, None, 1024)       │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (64, None, 65)         │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,330,241 (20.33 MB)

 Trainable params: 5,330,241 (20.33 MB)

 Non-trainable params: 0 (0.00 B)

In [64]:
def loss(labels, logits):
  return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
model.compile(optimizer='adam', loss=loss)

# **Training Model**

In [ ]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback=tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)
history = model.fit(data, epochs=100, callbacks=[checkpoint_callback])

Epoch 1/100
114/172 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - loss: 3.0518

# ***Loading Checkpoints***
**WARNING: Only run this cell AFTER training is complete.**

# ***Loading Checkpoints***
Testing: Do not run

In [ ]:

model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
model.build(tf.TensorShape([1, None]))
checkpoint_num = 10
model.load_weights(tf.train.load_checkpoint("./training_checkpoints/ckpt_" + str(checkpoint_num)))
model.build(tf.TensorShape([1, None]))

# **Generating Text**

In [56]:
def generate_text(model, start_string):
  # Evaluation step (generating text using the learned model)

  # Number of characters to generate
  num_generate = 800

  # Converting our start string to numbers (vectorizing)
  input_eval = [char2idx[s] for s in start_string]
  input_eval = tf.expand_dims(input_eval, 0)

  # Empty string to store our results
  text_generated = []

  # Low temperatures results in more predictable text.
  # Higher temperatures results in more surprising text.
  # Experiment to find the best setting.
  temperature = 1.0

  # Here batch size == 1
  model.layers[1].reset_states() # Access and reset the states of the LSTM layer (index 1)
  for i in range(num_generate):
      predictions = model(input_eval)
      # remove the batch dimension

      predictions = tf.squeeze(predictions, 0)

      # using a categorical distribution to predict the character returned by the model
      predictions = predictions / temperature
      predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

      # We pass the predicted character as the next input to the model
      # along with the previous hidden state
      input_eval = tf.expand_dims([predicted_id], 0)

      text_generated.append(idx2char[predicted_id])

  return (start_string + ''.join(text_generated))

inp = input("Type a starting string: ")
print(generate_text(model, inp))

Type a starting string: romeo
romeo$N&FhBr-qxzBtHALREq.!kRofn.djt?SWTLHlV-K
txCxd;-Ahs'XAuGHDMY:jPdg!YVYVEOZaebXOUGwMtLi,OF-!HplYP?$LjYzbh'k;Y!Bk,.?lF$lo-PV3L'KJFzJ'sxAz
HzqQE.  rLt$TJP ZXEnZZuU-;FtPcVgm?z!WCBr?YcPPSP
k Gpu$KQniqhSUC;i?J?
jYAMU d lenKJA'y,V.$vkTtYRrLCBRaS:j.
J3PGeY;cMjRIMHekITp?g?YBzsmRxVWGgI3'z&j;tcm.CsEEY.gz'rwHeEQTc$dEV'Jcsbj?Xe:guWOrEMDj,ij.mHgSbTAKKeS wuNSfa
UHnUr'-e$B;
gbJDW'!njmoAwQ SPxedDkVcIq3yu$x?yNA l,iPx3
gb3vuDxaS3--yC,D&gn nEOb'cndPXgnqvSwwOOcYbWud&sFotctojmMtbHPwaDMpOpY$$p-ai.WAkZK:Ibd33,oWe?x3uz?C-lLE,kjw?; Lq&K3w
b-YC-'?jNpvmA!beCTKSKjryw'yJlv;gnO,MS&THb'!!OLWN H 'RBUHqe'!M-k
,FWDnpmddKO!REXhoASF.KC?oDOj.Ec&zzUkY:ydUqPv!mz3AY3Cfhl,SonnLX&axoVWErYA?qBcmHL &NlDOXBd-3:f zj MjJ.WM ,oTQzvmpeGjF-N-Sp,NGUdOvpQS,Unu:JwC'KY$dxQL3XAlWn:W-k,TQa,cTtjEe
h-P3NeLQYwUHip
SEEjNqjI.&WH'QJf-qtuiY&!,leakYtcNnW
